'''
This notebook is intended to create the dataset required for clustering resistance project. 
There are three features: frequency, g-score and proximity to nearest non-self R mutation.
1. Frequency is derived from WHO catalogue: (Present_R)/(Present_R+Present_S). These are column names from the WHO catalogue, each column has a integer number denoting how many times each residue was found
in a resistant isolate or a susceptible isolate. Frequency is denoting the number of times a residue is found in resistant isolates compared to total number of appearances.
2. G-score: add the g-score precomputed to this dataset. Adjust for rpob gene.
3. proximity score: we use the 3D distance maps to calculate the proximity score to the nearest non-self R residue. In case of rpoB, we need to adjust the offset between PDB and WHO indices.
'''

In [94]:
import pandas as pd
import numpy as np
import re
import csv
import os
from evcouplings.compare import DistanceMap

## generate 3-to-1 letter AA codes

In [95]:
# check if the genes of interest are in catalog df
protein_sequences_file = '../data/catalog/protein_sequences.csv'
protein_sequences_df = pd.read_csv(protein_sequences_file)
# Extract genes of interest
genes_of_interest = protein_sequences_df['gene'].unique()
## load the 2021 complete excel file
who_catalog_path = '../data/catalog/WHO-UCN-GTB-PCI-2021.7-eng.xlsx'  # Update the path as necessary
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Mutation_catalogue',header=1)

In [96]:
# Flatten the column names if they are multi-level
if isinstance(who_catalog.columns, pd.MultiIndex):
    who_catalog.columns = ['_'.join(filter(None, col)).strip() for col in who_catalog.columns.values]

# Step 1: Read the file and flatten multi-index columns if present
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Mutation_catalogue', header=[0, 1])  # Use header=[0, 1] to capture multi-index columns

# Step 2: Flatten the multi-level columns
who_catalog.columns = ['_'.join(filter(None, col)).strip() for col in who_catalog.columns]

# Step 3: Check the flattened columns
print(who_catalog.columns)

# Step 1: Select the columns with the correct names
catalog_df = who_catalog[[
    'drug_Unnamed: 0_level_1', 
    'variant (common_name)_Unnamed: 2_level_1', 
    'DATASET_Present_R', 
    'DATASET_Present_S', 
    'FINAL CONFIDENCE GRADING_Unnamed: 51_level_1'
]]

# Step 2: Rename the columns to more meaningful names (optional)
catalog_df.columns = ['drug', 'variant', 'Present_R', 'Present_S', 'FINAL_CONFIDENCE_GRADING']

# Step 3: Check the first few rows of the selected data
print(catalog_df.head())
# Extract relevant columns

# Filter out rows with NaN values in the 'drug' or 'variant' columns
catalog_df = catalog_df.dropna(subset=['drug', 'variant'])

# subset for target genes
filtered_df = catalog_df[catalog_df['variant'].str.startswith(tuple(genes_of_interest))]


Index(['drug_Unnamed: 0_level_1', 'tier_Unnamed: 1_level_1',
       'variant (common_name)_Unnamed: 2_level_1',
       'Genome position_Unnamed: 3_level_1', 'DATASET_algorithm pass',
       'DATASET_Present_SOLO_R', 'DATASET_Present_SOLO_SR',
       'DATASET_Present_S', 'DATASET_Absent_S', 'DATASET_Present_R',
       'DATASET_Absent_R', 'DATASET_PPV', 'DATASET_PPV_lb', 'DATASET_PPV_ub',
       'DATASET_PPV | SOLO', 'DATASET_PPV | SOLO_lb', 'DATASET_PPV | SOLO_ub',
       'DATASET_Sensitivity', 'DATASET_Sensitivity_lb',
       'DATASET_Sensitivity_ub', 'DATASET_Specificity',
       'DATASET_Specificity_lb', 'DATASET_Specificity_ub', 'DATASET_LR+',
       'DATASET_LR+_lb', 'DATASET_LR+_ub', 'DATASET_LR-', 'DATASET_LR-_lb',
       'DATASET_LR-_ub', 'DATASET_OR', 'DATASET_OR_lb', 'DATASET_OR_ub',
       'DATASET_OR SOLO', 'DATASET_OR SOLO_lb', 'DATASET_OR SOLO_ub',
       'DATASET_OR SOLO_FE-sig', 'DATASET_Neutral masked',
       'INITIAL CONFIDENCE GRADING_Unnamed: 37_level_1',
       'DA

In [97]:
# Display the filtered DataFrame
print("number of mutations before discarding invalid residues", len(filtered_df))
# Discard rows that have 'ins' or 'del' in the variant column
filtered_df = filtered_df[~filtered_df['variant'].str.contains('ins|del')]
filtered_df=filtered_df.rename(columns={"FINAL_CONFIDENCE_GRADING": "confidence"})

number of mutations before discarding invalid residues 6576


In [98]:
## apply the 3-to-1 conversion
filtered_df['one_letter_mutation'] = filtered_df['variant'].str.extract(r'_(\w+\d+\w+)')
filtered_df['gene'] = filtered_df['variant'].str.split('_').str[0]
# Filter rows that match the valid mutation pattern (e.g., "A123B")
valid_pattern = r'^\w+_[A-Z]\d+[A-Z]$'
filtered_df = filtered_df[filtered_df['variant'].str.match(valid_pattern)]

## drop invalid mutations
## checked all variants to ensure they match the format A321B.
filtered_df = filtered_df.dropna(subset=['one_letter_mutation'])
filtered_df =filtered_df.drop_duplicates()

In [99]:
filtered_df.head()

,drug,variant,Present_R,Present_S,confidence,one_letter_mutation,gene
1576,BDQ,Rv0678_A36V,2.0,0.0,3) Uncertain significance,A36V,Rv0678
1581,BDQ,Rv0678_A99P,1.0,0.0,3) Uncertain significance,A99P,Rv0678
1582,BDQ,Rv0678_F19S,1.0,0.0,3) Uncertain significance,F19S,Rv0678
1583,BDQ,Rv0678_G121R,1.0,0.0,3) Uncertain significance,G121R,Rv0678
1584,BDQ,Rv0678_I67L,1.0,0.0,3) Uncertain significance,I67L,Rv0678


In [100]:
#check for unique genes
print(np.unique(filtered_df['gene']))

['Rv0678' 'atpE' 'ddn' 'embB' 'ethA' 'fgd1' 'gid' 'gyrA' 'gyrB' 'inhA'
 'katG' 'pepQ' 'pncA' 'rplC' 'rpoB' 'rpsL' 'tlyA']


In [101]:
# Desired column order
first_cols = ['drug', 'gene', 'variant']

# Get all other columns that are not in first_cols
remaining_cols = [col for col in filtered_df.columns if col not in first_cols]

# Reorder the DataFrame
filtered_df = filtered_df[first_cols + remaining_cols]
print("NUmber of mutations after filtering", len(filtered_df))

NUmber of mutations after filtering 4690


In [102]:
filtered_df.to_csv('../data/derived_features/mutations_with_one_letter_all_confidence.csv', index=False)

## feature 1: frequency

In [103]:
# Calculate the frequency of each variant in the population
filtered_df['frequency'] = (filtered_df['Present_R']) / (filtered_df['Present_R'] + filtered_df['Present_S'])
filtered_df.to_csv('../data/derived_features/all_proteins_frequency_catalog.csv', index=False)
filtered_df.head()

,drug,gene,variant,Present_R,Present_S,confidence,one_letter_mutation,frequency
1576,BDQ,Rv0678,Rv0678_A36V,2.0,0.0,3) Uncertain significance,A36V,1.0
1581,BDQ,Rv0678,Rv0678_A99P,1.0,0.0,3) Uncertain significance,A99P,1.0
1582,BDQ,Rv0678,Rv0678_F19S,1.0,0.0,3) Uncertain significance,F19S,1.0
1583,BDQ,Rv0678,Rv0678_G121R,1.0,0.0,3) Uncertain significance,G121R,1.0
1584,BDQ,Rv0678,Rv0678_I67L,1.0,0.0,3) Uncertain significance,I67L,1.0


## feature 2: proximity to nearest r-conferring mutations

### helper codes

In [ ]:
def extract_position_from_mutation(mutation):
    ## to extract position from mutation column
    match = re.search(r'\d+', mutation)
    if match:
        return int(match.group(0))
    return None


def adjust_number(index):
    # adjust index offset between pdb and who indices for rpob
    return index + 6 


# Map confidence levels to phenotypes
def map_confidence_to_phenotype(confidence):
    if confidence in ['1) Assoc w R', '2) Assoc w R - Interim']:
        return 'Resistant'
    elif confidence in ['4) Not assoc w R - Interim', '5) Not assoc w R']:
        return 'Susceptible'
    else:
        return 'Unknown'

### compute proximity

In [106]:
#load the derived dataframe from previous step
catalog_df=pd.read_csv('../data/derived_features/all_proteins_frequency_catalog.csv')
catalog_df['phenotype'] =catalog_df['confidence'].apply(map_confidence_to_phenotype)
print("distribution of phenotype", catalog_df['phenotype'].value_counts())
# extract position from mutation and create a column
catalog_df['position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
#Get the unique combinations of gene and drug
unique_genes_drugs = catalog_df[['gene', 'drug']].drop_duplicates()

# Print each gene and drug pair - debug step
for index, row in unique_genes_drugs.iterrows():
    print(f"Gene: {row['gene']}, Drug: {row['drug']}")
#create list of unique gene names
gene_names=list(np.unique(catalog_df['gene']))

distribution of phenotype phenotype
Unknown        4315
Resistant       315
Susceptible      60
Name: count, dtype: int64
Gene: Rv0678, Drug: BDQ
Gene: atpE, Drug: BDQ
Gene: pepQ, Drug: BDQ
Gene: tlyA, Drug: CAP
Gene: Rv0678, Drug: CFZ
Gene: pepQ, Drug: CFZ
Gene: ddn, Drug: DLM
Gene: fgd1, Drug: DLM
Gene: embB, Drug: EMB
Gene: ethA, Drug: ETH
Gene: inhA, Drug: ETH
Gene: katG, Drug: INH
Gene: inhA, Drug: INH
Gene: gyrA, Drug: LEV
Gene: gyrB, Drug: LEV
Gene: rplC, Drug: LZD
Gene: gyrA, Drug: MXF
Gene: gyrB, Drug: MXF
Gene: pncA, Drug: PZA
Gene: rpoB, Drug: RIF
Gene: rpsL, Drug: STM
Gene: gid, Drug: STM


In [107]:
##load the protein details file - we get the dist map id from it
protein_details_path = '../data/catalog/17_proteins_details.xlsx'  # Update the path as necessary
protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')
# Filter the DataFrame for the genes of interest
filtered_df = protein_details[protein_details['gene_name'].str.contains('|'.join(gene_names), case=False, na=False)]
print("previewing protein details file")
filtered_df.head()

previewing protein details file


,Drug,gene_name,filename,Uniprot,references,Entry,Protein names,rv,drug_full,Aliases,start_position_on_the_genomic_accession,end_position_on_the_genomic_accession,orientation
0,RIF,rpoB,rpoBC.fasta,RPOB_MYCTU,"WHO 2021 catalog, category 1",P9WGY9,DNA-directed RNA polymerase subunit beta (RNAP...,Rv0667,rifampicin,Rv0667,759806.0,763324.0,plus
1,INH,inhA,FabG1-inhA.fasta,INHA_MYCTU,"WHO 2021 catalog, category 1",P9WGR1,Enoyl-[acyl-carrier-protein] reductase [NADH] ...,Rv1484,isoniazid,Rv1484,1674201.0,1675010.0,plus
2,INH,katG,KatG.fasta,KATG_MYCTU,"WHO 2021 catalog, category 1",P9WIE5,Catalase-peroxidase (CP) (EC 1.11.1.21) (Perox...,Rv1908c,isoniazid,Rv1908c,2153889.0,2156111.0,minus
3,EMB,embB,embCAB.fasta,EMBB_MYCTU,"WHO 2021 catalog, category 1",P9WNL7,Probable arabinosyltransferase B (EC 2.4.2.-),Rv3795,ethambutol,Rv3795,4246513.0,4249809.0,plus
4,PZA,pncA,pncA.fasta,PNCA_MYCTU,"WHO 2021 catalog, category 1",I6XD65,Nicotinamidase/pyrazinamidase (Nicotinamidase)...,Rv2043c,pyrazinamide,Rv2043c,2288681.0,2289241.0,minus


In [112]:
# Initialize an empty list to store all processed DataFrames
all_gene_data = []


def non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, distance_map, gene_name):
    positions_causing_error = []
    proximity_to_nonself_r_conferring = []
    nearest_mutation_index = []

    proximity_to_nonself_r_conferring_1d = []
    nearest_mutation_index_1d = []

    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []
        distances_in_1d = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = distance_map.dist(current_position, r_pos, raise_na=True)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                        distances_in_1d.append((np.abs(current_position-r_pos), r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])

            # Get the distance in primary sequence to the nearest R mutation
            min_1d_distance, min_1d_index = min(distances_in_1d, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
            min_1d_distance, min_1d_index = np.nan, np.nan

        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
        proximity_to_nonself_r_conferring_1d.append(min_1d_distance)
        nearest_mutation_index_1d.append(min_1d_index)

    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_to_R_Conferring'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_Mutation_Index'] = nearest_mutation_index
    gene_subset['Proximity_to_R_Conferring_1D'] = proximity_to_nonself_r_conferring_1d
    gene_subset['Nearest_Mutation_Index_1D'] = nearest_mutation_index_1d
    return gene_subset

for index, row in filtered_df.iterrows():
    # Extract metadata from current row
    fasta_filename = row['filename']
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    drug_code = row['Drug']
    gene_name = row['gene_name']
    
    print(f"\nProcessing {gene_name} from file: {fasta_filename}")

    # Subset catalog to this gene
    gene_subset = catalog_df[catalog_df['gene'] == gene_name].drop_duplicates()

    # Load distance map
    distmap_path = f"../data/filtered_distmaps/{entry}"
    dist_map = DistanceMap.from_file(distmap_path)
    gene_length = dist_map.dist_matrix.shape[0]
    print(f"Distance map length for {gene_name}: {gene_length}")

    # Pull residues
    residues_df = dist_map.residues_i.copy()
    residues_df['coord_id'] = residues_df['coord_id'].astype(int)
    
    # Get WHO mutation info
    gene_subset = gene_subset.copy()
    #apply adjust number method if it is rpob
    if gene_name == 'rpoB':
        gene_subset['WHO_Adjusted_Position'] = gene_subset['position'].apply(adjust_number)
        #set the who adjusted position as r-conferring positions array for proximity calculation
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['WHO_Adjusted_Position'].tolist()
    else:
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['position'].tolist()
    
    unique_valid_r_positions = np.unique(r_conferring_positions)
    print("Number of Unique valid R-conferring positions:", len(unique_valid_r_positions))

    # Proximity analysis
    gene_subset = non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, dist_map, gene_name)

    # load and merge G-scores
    try:
        g_scores = pd.read_csv(f"../data/output/{uniprot}/G_scores.csv")
        print(f"G_scores loaded for {uniprot}")
        if gene_name == 'rpoB':
            # gene_subset['adjusted_position'] = gene_subset['position'].apply(adjust_number)
            merged = pd.merge(gene_subset, g_scores, left_on='WHO_Adjusted_Position', right_on='residue', how='left')
        else:
            #since for other genes we are not adjusting their position, we use the actual column
            merged = pd.merge(gene_subset, g_scores, left_on='position', right_on='residue', how='left')

        merged.drop(columns=['residue', 'Unnamed: 0'], errors='ignore', inplace=True)

        all_gene_data.append(merged)

    except FileNotFoundError as e:
        print(f"Skipping G-score merge for {uniprot}: {e}")
        all_gene_data.append(gene_subset)



Processing rpoB from file: rpoBC.fasta
Distance map length for rpoB: 1138
Number of Unique valid R-conferring positions: 25
G_scores loaded for RPOB_MYCTU

Processing inhA from file: FabG1-inhA.fasta
Distance map length for inhA: 246
Number of Unique valid R-conferring positions: 1
G_scores loaded for INHA_MYCTU

Processing katG from file: KatG.fasta
Distance map length for katG: 716
Number of Unique valid R-conferring positions: 2
G_scores loaded for KATG_MYCTU

Processing embB from file: embCAB.fasta
Distance map length for embB: 1054
Number of Unique valid R-conferring positions: 7
G_scores loaded for EMBB_MYCTU

Processing pncA from file: pncA.fasta
Distance map length for pncA: 185
Number of Unique valid R-conferring positions: 82
G_scores loaded for PNCA_MYCTU

Processing gyrA from file: gyrBA.fasta
Distance map length for gyrA: 766
Number of Unique valid R-conferring positions: 4
G_scores loaded for GYRA_MYCTU

Processing rpsL from file: rpsL.fasta
Distance map length for rpsL:

In [113]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

In [114]:
final_df

,drug,gene,variant,Present_R,Present_S,confidence,one_letter_mutation,frequency,phenotype,position,WHO_Adjusted_Position,Proximity_to_R_Conferring,Nearest_Mutation_Index,Proximity_to_R_Conferring_1D,Nearest_Mutation_Index_1D,G_score
0,RIF,rpoB,rpoB_S450L,6536.0,74.0,1) Assoc w R,S450L,0.988805,Resistant,450,456.0,1.328860,455.0,1.0,455.0,2.808807
1,RIF,rpoB,rpoB_D435V,732.0,9.0,1) Assoc w R,D435V,0.987854,Resistant,435,441.0,1.324017,440.0,1.0,440.0,4.291423
2,RIF,rpoB,rpoB_H445Y,347.0,4.0,1) Assoc w R,H445Y,0.988604,Resistant,445,451.0,1.329035,452.0,1.0,450.0,16.133619
3,RIF,rpoB,rpoB_H445D,288.0,3.0,1) Assoc w R,H445D,0.989691,Resistant,445,451.0,1.329035,452.0,1.0,450.0,16.133619
4,RIF,rpoB,rpoB_D435Y,162.0,44.0,1) Assoc w R,D435Y,0.786408,Resistant,435,441.0,1.324017,440.0,1.0,440.0,4.291423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4685,DLM,fgd1,fgd1_L323F,0.0,4.0,3) Uncertain significance,L323F,0.000000,Unknown,323,NaN,NaN,NaN,NaN,NaN,NaN
4686,DLM,fgd1,fgd1_K183M,0.0,5.0,3) Uncertain significance,K183M,0.000000,Unknown,183,NaN,NaN,NaN,NaN,NaN,NaN
4687,DLM,fgd1,fgd1_R45C,0.0,7.0,3) Uncertain significance,R45C,0.000000,Unknown,45,NaN,NaN,NaN,NaN,NaN,NaN
4688,DLM,fgd1,fgd1_M93T,0.0,9.0,3) Uncertain significance,M93T,0.000000,Unknown,93,NaN,NaN,NaN,NaN,NaN,NaN


In [115]:
np.unique(final_df['confidence'])

array(['1) Assoc w R', '2) Assoc w R - Interim',
       '3) Uncertain significance', '4) Not assoc w R - Interim',
       '5) Not assoc w R'], dtype=object)

In [116]:
final_df.to_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv",index=False)

## debug code from ANNA

In [117]:
test_case=pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

In [118]:
## test cases

from Bio import SeqIO
from io import StringIO
import pandas as pd
import requests

def fetch_uniprot_sequence(uniprot_id):
    """
    Fetches a protein sequence from UniProt for a single accession ID.

    Parameters:
        uniprot_id (str): UniProt accession ID.

    Returns:
        SeqRecord: A Biopython SeqRecord object.
    """
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    response.raise_for_status()

    fasta_io = StringIO(response.text)
    record = next(SeqIO.parse(fasta_io, "fasta")) 

    return record


    
def verify_mutations_vs_uniprot(subset, entry, check_column):
    """
    checks for sequence match between WHO table and uniprot entry sequence

    subset: pd.DataFrame of WHO table
        position - column that contains the 1 indexed position
        original_AA - column with 1 letter AA code of WT AA

    entry: str
        uniprot entry
    """
    # fetch the sequence from uniprot
    record = fetch_uniprot_sequence(entry)
    print(entry, record.seq)
    mismatches=[]

    for idx,row in subset.iterrows():
        #position was showing float output
        position=int(row[check_column])
        WHO_aa=row['original_AA']

        #check uniprot sequence
        if record.seq[position -1] == WHO_aa:
            continue
        else:
            print(f"Error for protein {record.id}, WHO and Uniprot do not match for position {row[check_column]}. WHO AA: {row.original_AA}, Uniprot_AA:{record.seq[row.position -1]}")
            mismatches.append({
                "UniProt_ID": record.id,
                "Gene": row['gene'],
                "Position": int(row[check_column]),
                "WHO_AA": WHO_aa,
                "UniProt_AA": record.seq[position -1],
                "Phenotype": row['phenotype']
            })
    return mismatches


def run_test():
    catalog_df = pd.read_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv")

    catalog_df['raw_position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
    catalog_df['original_AA'] =catalog_df['one_letter_mutation'].str.extract(r'(\S{1})\d+')
    catalog_df['mutated_AA'] =catalog_df['one_letter_mutation'].str.extract(r'\d+(\S{1})')
    all_mismatches = []

    ##load the protein details file
    protein_details_path = '../data/catalog/17_proteins_details.xlsx' 
    protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

    output_df_list = []

    # Iterate through each protein
    for idx, row in protein_details.iterrows():
        entry = row.Entry
        gene = row.gene_name
        subset = catalog_df.query("gene==@gene")
        print(f"running debugging for gene: {gene}")
        if gene=='rpoB':
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="WHO_Adjusted_Position")
        else:
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="position")
        all_mismatches.extend(mismatches)
        
    # Save to CSV
    mismatch_df = pd.DataFrame(all_mismatches)
    mismatch_df.to_csv("../data/who_uniprot_mismatch_report.csv", index=False)
    print("Saved mismatch report to 'results/who_uniprot_mismatch_report.csv'")




In [119]:
run_test()


running debugging for gene: rpoB
P9WGY9 MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLREPLEVPGLLDVQTDSFEWLIGSPRWRESAAERGDVNPVGGLEEVLYELSPIEDFSGSMSLSFSDPRFDDVKAPVDECKDKDMTYAAPLFVTAEFINNNTGEIKSQTVFMGDFPMMTEKGTFIINGTERVVVSQLVRSPGVYFDETIDKSTDKTLHSVKVIPSRGAWLEFDVDKRDTVGVRIDRKRRQPVTVLLKALGWTSEQIVERFGFSEIMRSTLEKDNTVGTDEALLDIYRKLRPGEPPTKESAQTLLENLFFKEKRYDLARVGRYKVNKKLGLHVGEPITSSTLTEEDVVATIEYLVRLHEGQTTMTVPGGVEVPVETDDIDHFGNRRLRTVGELIQNQIRVGMSRMERVVRERMTTQDVEAITPQTLINIRPVVAAIKEFFGTSQLSQFMDQNNPLSGLTHKRRLSALGPGGLSRERAGLEVRDVHPSHYGRMCPIETPEGPNIGLIGSLSVYARVNPFGFIETPYRKVVDGVVSDEIVYLTADEEDRHVVAQANSPIDADGRFVEPRVLVRRKAGEVEYVPSSEVDYMDVSPRQMVSVATAMIPFLEHDDANRALMGANMQRQAVPLVRSEAPLVGTGMELRAAIDAGDVVVAEESGVIEEVSADYITVMHDNGTRRTYRMRKFARSNHGTCANQCPIVDAGDRVEAGQVIADGPCTDDGEMALGKNLLVAIMPWEGHNYEDAIILSNRLVEEDVLTSIHIEEHEIDARDTKLGAEEITRDIPNISDEVLADLDERGIVRIGAEVRDGDILVGKVTPKGETELTPEERLLRAIFGEKAREVRDTSLKVPHGESGKVIGIRVFSREDEDELPAGVNELVRVYVAQKRKISDGDKLAGRHGNKGVIGKILPVEDMPFLADGTPVDIILNTHGVPRRMNIGQILETHLGWCAHSGWKVDAAKGVPDWAARLP

In [120]:
# Load mismatch report and test case
our_df = pd.read_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv")
test_cases = pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

# Merge or filter mismatches that match the test case variants
check_variants = test_cases['variant'].unique()

test_columns = ["Nearest_Mutation_Index", "G_score",  "Proximity_to_R_Conferring"]

mismatches = []

for current_var in check_variants:
    ours = our_df.query("variant==@current_var")
    test = test_cases.query("variant==@current_var")

    for t in test_columns: 
        if ours[t].values[0] != test[t].values[0]:
            print("error for", current_var, "in column", t)

            mismatches.append(current_var)


if len(mismatches) == 0:
    print("All test cases passed!")

else: 
    print("Error - Uniprot-WHO mapping failed for 1 or more proteins")



All test cases passed!
